# Workshop: Gemma from Scratch
## Notebook 6: RMSNorm and Normalization

**Estimated Time: 10 minutes**

Modern transformers like Gemma 2 and Gemma 3 use **Root Mean Square Layer Normalization (RMSNorm)**. It is simpler and faster than standard LayerNorm because it doesn't calculate the mean, only the variance.

### Learning Objectives:
1. Understand the math of RMSNorm.
2. **Implement the Gemma-specific "Add-One" RMSNorm layer.**
3. Learn about the Pre-norm and Post-norm positioning in the transformer block.

In [ ]:
import torch
import torch.nn as nn
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

dim = 128
eps = 1e-6

### 1. RMSNorm Math

Standard LayerNorm: $y = \frac{x - E[x]}{\sqrt{Var[x] + \epsilon}} \cdot \gamma + \beta$

RMSNorm: $y = \frac{x}{\sqrt{Mean(x^2) + \epsilon}} \cdot \gamma$

**The Gemma Twist:** Gemma's implementation initializes the weight to zero and uses $(1 + \gamma)$ as the scaling factor. This ensures that at the start of training, the layer performs a simple identity scaling, which is more stable.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # Gemma initialization: weight starts at zero
        self.weight = nn.Parameter(torch.zeros(dim))

    def _norm(self, x):
        # x: (..., dim)
        # 1. Compute RMS: sqrt(mean(x^2))
        # 2. Divide x by RMS
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        # Apply the (1 + weight) scaling factor used in Gemma
        return self._norm(x.float()).type_as(x) * (1.0 + self.weight)

norm = RMSNorm(dim)
x = torch.randn(2, 5, dim) * 10 # High variance input
out = norm(x)

print("Input variance:", x.var().item())
print("Output variance:", out.var().item())
print("Output mean (not zero, unlike LayerNorm):", out.mean().item())

# Check assertion
assert out.shape == x.shape
print("✅ Success! Output shape matches input shape.")

### 2. Positioning in the Block

Gemma 2 and 3 use **Hybrid Normalization**. This means the input to the attention and MLP sub-layers is normalized *before* and *after* the operations.

This "double-norm" is a unique architectural choice that helps stabilize the training of deep models.

### Exercise:
Why does initializing the weight to zero and using `(1 + weight)` help with training stability compared to initializing the weight to one and using it directly?

<details>
<summary><b>Click to see solution</b></summary>

By initializing to zero and using `(1 + weight)`, the layer effectively starts as an **Identity** operation (it scales by exactly 1.0). This means the gradients can flow easily through the network at the very beginning of training without being skewed by random initial scaling values. It allows the model to find a stable path before the normalization parameters start making significant adjustments.
</details>